# Ingestion Pipeline

End-to-end pipeline that:
1. Loads `.txt` knowledge-base files
2. Parses metadata from file headers
3. Chunks the documents
4. Embeds the chunks
5. Stores them in the vector database
6. Runs a sanity-check retrieval query

**Reuses** logic from `1_load_chunk.ipynb`, `2_embeddings.ipynb`, `3_retrieval.ipynb`, `4_basic_RAG.ipynb` and `src/` py files.

## Setup 
### Imports & Paths

In [1]:
# !pip install -r ../requirements.txt

In [2]:
from pathlib import Path
import sys
import os
import numpy as np
from dotenv import load_dotenv

# ── Project root & src on path ──────────────────────────────────────────────
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# ── Knowledge-base directories ───────────────────────────────────────────────
LGBT_EU_BY_COUNTRY_DIR = PROJECT_ROOT / "data" / "3_txt_KB" / "LGBT_EU" / "by_country"
LGBT_EU_BY_SUBSET_DIR  = PROJECT_ROOT / "data" / "3_txt_KB" / "LGBT_EU" / "by_subset"
HIV_KB_DIR             = PROJECT_ROOT / "data" / "3_txt_KB" / "HIV_AIDS_data"
UNICEF_KB_DIR          = PROJECT_ROOT / "data" / "3_txt_KB" / "UNICEF_Immunization"
FRA_KB_DIR             = PROJECT_ROOT / "data" / "3_txt_KB" / "PDF_Reports" / "FRA"
ILGA_KB_DIR            = PROJECT_ROOT / "data" / "3_txt_KB" / "PDF_Reports" / "ILGA"

KB_DIRS = [
    LGBT_EU_BY_COUNTRY_DIR,
    LGBT_EU_BY_SUBSET_DIR,
    HIV_KB_DIR,
    UNICEF_KB_DIR,
    FRA_KB_DIR,
    ILGA_KB_DIR,
]

TEST_RANGE_K_DIR       = PROJECT_ROOT / "data" / "4_testing" / "range_of_k"
TEST_RANGE_CHUNK_DIR   = PROJECT_ROOT / "data" / "4_testing" / "range_of_chunk"
print("Project root:", PROJECT_ROOT)
for d in KB_DIRS:
    status = "✓" if d.exists() else "✗ (not found)"
    print(f"  {status}  {d.relative_to(PROJECT_ROOT)}")

Project root: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG
  ✓  data\3_txt_KB\LGBT_EU\by_country
  ✓  data\3_txt_KB\LGBT_EU\by_subset
  ✓  data\3_txt_KB\HIV_AIDS_data
  ✓  data\3_txt_KB\UNICEF_Immunization
  ✓  data\3_txt_KB\PDF_Reports\FRA
  ✓  data\3_txt_KB\PDF_Reports\ILGA


In [3]:
# ── Standard library & third-party ───────────────────────────────────────────
from typing import Dict, List, Tuple

from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

# ── src modules (reused from previous notebooks) ─────────────────────────────
from vectorstore import build_vectorstore
from retrieval   import retrieve, print_results, format_context
from llm         import build_prompt, ask


In [4]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("API key loaded:", "✅" if OPENAI_API_KEY else "❌ NOT FOUND")

API key loaded: ✅


In [5]:
# used in evaluation stage
import nltk
import re
import numpy as np
import nltk
from sklearn.metrics.pairwise import cosine_similarity
from textstat import flesch_reading_ease

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from pathlib import Path
import importlib, subprocess, sys # used for dependency management in eval

## Parameters

- uses `RecursiveCharacterTextSplitter` used in `1_load_chunk.ipynb`.
- Edit params to change how it will be applied.

In [ ]:
# whether we actually use OpenAI credits on this
USE_OPENAI = True

# ── Chunking parameters (mirrors 1_load_chunk.ipynb) ─────────────────────────
# Note: used for the main pipeline. Later on we use sweep functions to test ranges. See the Parameter Sweep section.
CHUNK_SIZE    = 750
CHUNK_OVERLAP = 75

# ── Vector-store persistence path ─────────────────────────────────────────────
VECTORSTORE_DIR = PROJECT_ROOT / "data" / "vectorstore"

# ── Embedding model (mirrors 2_embeddings.ipynb) ──────────────────────────────
EMBEDDING_MODEL = "text-embedding-3-small"

# ── Retrieval parameters (mirrors 3_retrieval.ipynb) ─────────────────────────
# TOP_K: number of chunks returned per query. Tune this to balance coverage vs noise.
# Sweep range is tested in the Parameter Sweep section below.
TOP_K = 5

print(f"CHUNK_SIZE={CHUNK_SIZE}, CHUNK_OVERLAP={CHUNK_OVERLAP}")
print(f"EMBEDDING_MODEL={EMBEDDING_MODEL}")
print(f"VECTORSTORE_DIR={VECTORSTORE_DIR}")

CHUNK_SIZE=500, CHUNK_OVERLAP=50
EMBEDDING_MODEL=text-embedding-3-small
VECTORSTORE_DIR=C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\vectorstore


## Load All `.txt` Files
- load in from knowledge base (the output of `7_generate_text_knowledge_base.ipynb`)

In [7]:
def collect_txt_files(directories: List[Path]) -> List[Path]:
    """Recursively collect every .txt file from the given directories."""
    files: List[Path] = []
    for directory in directories:
        if not directory.exists():
            print(f"  ⚠  Directory not found, skipping: {directory}")
            continue
        found = sorted(directory.rglob("*.txt"))
        print(f"  Found {len(found):>4} files in {directory.relative_to(PROJECT_ROOT)}")
        files.extend(found)
    return files


all_txt_files = collect_txt_files(KB_DIRS)
print(f"\nTotal .txt files: {len(all_txt_files)}")

  Found 4427 files in data\3_txt_KB\LGBT_EU\by_country
  Found  699 files in data\3_txt_KB\LGBT_EU\by_subset
  Found   69 files in data\3_txt_KB\HIV_AIDS_data
  Found  292 files in data\3_txt_KB\UNICEF_Immunization
  Found   69 files in data\3_txt_KB\PDF_Reports\FRA
  Found  171 files in data\3_txt_KB\PDF_Reports\ILGA

Total .txt files: 5727


## Handle Metadata + Content

In [8]:
def parse_document(file_path: Path) -> Tuple[str, Dict[str, str]]:
    raw = file_path.read_text(encoding="utf-8")
    lines = raw.splitlines()

    metadata: Dict[str, str] = {}
    content_start = 0

    for i, line in enumerate(lines):
        stripped = line.strip()

        if stripped == "":          # blank line → header ends here
            content_start = i + 1
            break

        if ":" in stripped:         # metadata line  KEY: value
            key, _, value = stripped.partition(":")
            metadata[key.strip().lower()] = value.strip()
        else:
            # Not a metadata line and not blank → no header, treat whole file as content
            content_start = 0
            metadata = {}
            break

    content = "\n".join(lines[content_start:]).strip()
    metadata["Source"] = str(file_path)

    return content, metadata

In [9]:
# ── Quick smoke-test on the first available file ──────────────────────────────
if all_txt_files:
    _sample_content, _sample_meta = parse_document(all_txt_files[0])
    print("Sample file :", all_txt_files[0].name)
    print("Metadata    :", _sample_meta)
    print("Content (100 chars):", _sample_content[:100], "...")
else:
    print("No files found — check KB_DIRS above.")

Sample file : b1_a_answer_by_Austria.txt
Metadata    : {'dataset': 'EU_LGBT', 'question_code': 'b1_a', 'subset': 'Austria', 'Source': 'C:\\Users\\RAZER\\Desktop\\portfolio-projects\\1. RAG\\data\\3_txt_KB\\LGBT_EU\\by_country\\LGBT_Survey_DailyLife\\b1_a_answer_by_Austria.txt'}
Content (100 chars): Question b1_a — In your opinion, how widespread is offensive language about lesbian, gay, bisexual a ...


In [10]:
def build_documents(file_paths: List[Path]) -> List[Document]:
    """
    Parse every file and return a list of LangChain Documents.
    Mirrors the Document creation pattern from 1_load_chunk.ipynb.
    """
    docs: List[Document] = []
    errors: List[str] = []

    for fp in file_paths:
        try:
            content, metadata = parse_document(fp)
            if content:            # skip empty files
                docs.append(Document(page_content=content, metadata=metadata))
        except Exception as exc:
            errors.append(f"{fp.name}: {exc}")

    if errors:
        print(f"⚠  {len(errors)} file(s) could not be parsed:")
        for e in errors:
            print("  ", e)

    print(f"\nDocuments created : {len(docs)}")
    return docs


raw_documents = build_documents(all_txt_files)


Documents created : 5727


## Chunk Documents

Reuses the `RecursiveCharacterTextSplitter` we made `1_load_chunk.ipynb`.

In [11]:
# ── Text splitter — mirrors 1_load_chunk.ipynb ────────────────────────────────
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    add_start_index=True,   # keeps track of character offset (used in 1_load_chunk.ipynb)
)

chunks = text_splitter.split_documents(raw_documents)

print(f"Raw documents : {len(raw_documents)}")
print(f"Chunks        : {len(chunks)}")
print(f"Avg chunk size: {sum(len(c.page_content) for c in chunks) // max(len(chunks), 1)} chars")

Raw documents : 5727
Chunks        : 34809
Avg chunk size: 362 chars


In [12]:
# ── Inspect a sample chunk ────────────────────────────────────────────────────
if chunks:
    sample = chunks[0]
    print("Sample chunk metadata :", sample.metadata)
    print("Sample chunk content  :", sample.page_content[:200], "...")

Sample chunk metadata : {'dataset': 'EU_LGBT', 'question_code': 'b1_a', 'subset': 'Austria', 'Source': 'C:\\Users\\RAZER\\Desktop\\portfolio-projects\\1. RAG\\data\\3_txt_KB\\LGBT_EU\\by_country\\LGBT_Survey_DailyLife\\b1_a_answer_by_Austria.txt', 'start_index': 0}
Sample chunk content  : Question b1_a — In your opinion, how widespread is offensive language about lesbian, gay, bisexual and/or transgender people by politicians in the country where you live? | Austria responses (Bisexual ...


## Embed Documents
- Reuses the `OpenAIEmbeddings` setup from `2_embeddings.ipynb`.

In [13]:
# ── Embedding model — mirrors 2_embeddings.ipynb ──────────────────────────────
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Quick sanity check: embed a single string
_test_vec = embeddings.embed_query("test")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Vector dimension: {len(_test_vec)}")

Embedding model : text-embedding-3-small
Vector dimension: 1536


## Store in Vector DB
- Reuses `load_vectorstore` from `src/vectorstore.py`.

In [14]:
# ── Persist directory ─────────────────────────────────────────────────────────
VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Vector-store directory: {VECTORSTORE_DIR}")

Vector-store directory: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\vectorstore


In [15]:
# ── Build / overwrite the vector store ───────────────────────────────────────
# load_vectorstore is expected to accept (chunks, embeddings, persist_directory)
# and return a Chroma (or equivalent) vectorstore — as used in 3_retrieval.ipynb

vectorstore = build_vectorstore(
    documents=chunks,
    embeddings=embeddings,
    persist_directory=str(VECTORSTORE_DIR),
)

print(f"\n✓ Vector store built and persisted to: {VECTORSTORE_DIR}")
print(f"  Total vectors stored: {vectorstore._collection.count()}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Built vectorstore: 363004 chunks

✓ Vector store built and persisted to: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\vectorstore
  Total vectors stored: 363004


C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\src\vectorstore.py:32: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


# Testing
## Small test of End-to-End Retrieval
- Runs a sample query through the full pipeline
- Same structure as `4_basic_RAG.ipynb`.

In [16]:
SAMPLE_QUERY = "Bisexual women daily life experience in Romania"
print(f"Sample query: {SAMPLE_QUERY}")

Sample query: Bisexual women daily life experience in Romania


### 1. Retrieve relevant chunks


In [17]:
# retrieve() mirrors 3_retrieval.ipynb usage
results = retrieve(
    query=SAMPLE_QUERY,
    vectorstore=vectorstore,
    k=TOP_K,
)

print(f"Retrieved {len(results)} chunks:\n")
print_results(query = SAMPLE_QUERY, results = results)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieved 5 chunks:

Query: 'Bisexual women daily life experience in Romania'

--- Result 1 ---
Source: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\3_txt_KB\LGBT_EU\by_subset\LGBT_Survey_DailyLife\g4_b_answer_by_Bisexualwomen.txt
Question g4_b — You have been treated with less respect than other people - In the last six months, in your day-to-day life, how often have any of the following things happened to you because you are or are assumed to be lesbian, gay, bisexual and/or transgender? | Bisexual women responses in Romania: Never happened in the last sixth months: 59%, Happened only once in the last six months: 16%, 2-5 times in the last six months: 14%, 6 times or more in the last six months: 4%, Don`t know: 7%

--- Result 2 ---
Source: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\3_txt_KB\LGBT_EU\by_subset\LGBT_Survey_DailyLife\g4_b_answer_by_Bisexualwomen.txt
Question g4_b — You have been treated with less respect than other people - In the last six months, in 

### 2. Format context

In [18]:
context = format_context(results)
print("Context passed to LLM (first 500 chars):")
print(context[:500], "...")

Context passed to LLM (first 500 chars):
Question g4_b — You have been treated with less respect than other people - In the last six months, in your day-to-day life, how often have any of the following things happened to you because you are or are assumed to be lesbian, gay, bisexual and/or transgender? | Bisexual women responses in Romania: Never happened in the last sixth months: 59%, Happened only once in the last six months: 16%, 2-5 times in the last six months: 14%, 6 times or more in the last six months: 4%, Don`t know: 7%

Ques ...


### 3. Build prompt

In [19]:
# build_prompt() mirrors 4_basic_RAG.ipynb usage
prompt = build_prompt(query=SAMPLE_QUERY, context=context)
print("Prompt (first 500 chars):")
print(prompt[:500], "...")

Prompt (first 500 chars):
[SystemMessage(content="You are a helpful assistant. Answer the user's question using only the context provided below. If the answer is not in the context, say 'I don't have enough information to answer that.'", additional_kwargs={}, response_metadata={}), HumanMessage(content='Context:\nQuestion g4_b — You have been treated with less respect than other people - In the last six months, in your day-to-day life, how often have any of the following things happened to you because you are or are assumed to be lesbian, gay, bisexual and/or transgender? | Bisexual women responses in Romania: Never happened in the last sixth months: 59%, Happened only once in the last six months: 16%, 2-5 times in the last six months: 14%, 6 times or more in the last six months: 4%, Don`t know: 7%\n\nQuestion g4_b — You have been treated with less respect than other people - In the last six months, in your day-to-day life, how often have any of the following things happened to you bec

### 4. Ask the LLM

In [20]:
# ask() mirrors 4_basic_RAG.ipynb usage
answer = ask(prompt, context)
print("=" * 60)
print("QUESTION:", SAMPLE_QUERY)
print("=" * 60)
print("ANSWER:")
print(answer)

QUESTION: Bisexual women daily life experience in Romania
ANSWER:
In the last six months, the daily life experiences of bisexual women in Romania regarding being treated with less respect due to their sexual orientation or assumed identity are as follows: 59% reported that they have never experienced this, 16% said it happened only once, 14% experienced it 2-5 times, 4% experienced it 6 times or more, and 7% did not know.


## Pipeline Summary

In [21]:
print("Pipeline complete ✓")
print(f"  Files loaded    : {len(all_txt_files)}")
print(f"  Documents parsed: {len(raw_documents)}")
print(f"  Chunks created  : {len(chunks)}")
print(f"  Vectors stored  : {vectorstore._collection.count()}")
print(f"  Vector store at : {VECTORSTORE_DIR}")

Pipeline complete ✓
  Files loaded    : 5727
  Documents parsed: 5727
  Chunks created  : 34809
  Vectors stored  : 363004
  Vector store at : C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\vectorstore


# Metadata Filtering
Routes each query to the most relevant data source(s) based on its content, then filters retrieval accordingly.

- Policy / country comparisons / legal rights → **ILGA** (primary) + CSV sources as backup
- Survey data / methodology / statistics / discrimination / lived experience → **FRA** (primary) + CSV sources as backup
    - HIV/AIDS specific → HIV dataset
    - Immunization / vaccination → UNICEF dataset
    - Fallback: search all sources


## Setup

In [22]:
# Source-tag constants — must match the 'dataset' or directory metadata set during ingestion
SOURCE_FRA  = "FRA"
SOURCE_ILGA = "ILGA"
SOURCE_LGBT_COUNTRY = "EU_LGBT"   # by_country files
SOURCE_LGBT_SUBSET  = "EU_LGBT"   # by_subset files (same dataset tag)
SOURCE_HIV   = "HIV"
SOURCE_UNICEF = "UNICEF"

# Keyword sets used for routing
_POLICY_KEYWORDS = {
    "policy", "law", "legal", "legislation", "rights", "protection",
    "country", "countries", "compare", "comparison", "ranking",
    "rainbow", "marriage", "adoption", "ban", "criminalise", "criminaliz",
    "parliament", "government", "eu", "europe", "european",
}
_SURVEY_KEYWORDS = {
    "survey", "data", "study", "research", "report", "percentage", "percent",
    "discrimination", "harassment", "violence", "experience", "feel", "comfortable",
    "open", "identity", "trans", "transgender", "bisexual", "lesbian", "gay",
    "queer", "lgbti", "lgbtq", "daily life", "work", "hate", "crime",
}
_HIV_KEYWORDS    = {"hiv", "aids", "antiretroviral", "art", "prevalence", "hiv/aids"}
_UNICEF_KEYWORDS = {"vaccine", "vaccination", "immunization", "immunisation",
                    "measles", "dtp", "dtp3", "polio", "coverage"}




## Implementation

In [23]:
# Looks at a query, returns a Chroma metadata filter dict.
def route_query_to_sources(query):
    """
    Priority order:
      1. HIV / UNICEF keywords  → hard-target those datasets only
      2. Policy keywords        → ILGA primary + LGBT CSV backup
      3. Survey/experience kws  → FRA primary + LGBT CSV backup
      4. No strong signal       → all sources (no filter)
    """
    tokens = set(query.lower().split())

    if tokens & _HIV_KEYWORDS:
        sources = [SOURCE_HIV]
        label   = "HIV dataset (specific HIV/AIDS query)"
    elif tokens & _UNICEF_KEYWORDS:
        sources = [SOURCE_UNICEF]
        label   = "UNICEF dataset (immunization query)"
    elif tokens & _POLICY_KEYWORDS:
        # ILGA primary + LGBT CSV as supplementary facts
        sources = [SOURCE_ILGA, SOURCE_LGBT_COUNTRY, SOURCE_LGBT_SUBSET]
        label   = "ILGA (policy/rights query) + LGBT CSV backup"
    elif tokens & _SURVEY_KEYWORDS:
        # FRA primary + LGBT CSV as supplementary facts
        sources = [SOURCE_FRA, SOURCE_LGBT_COUNTRY, SOURCE_LGBT_SUBSET]
        label   = "FRA (survey/experience query) + LGBT CSV backup"
    else:
        sources = None
        label   = "all sources (no strong signal)"

    print(f"  → Routing to: {label}")

    if sources is None:
        return None   # no filter

    if len(sources) == 1:
        return {"dataset": {"$eq": sources[0]}}
    else:
        return {"dataset": {"$in": sources}}

In [24]:
# ── Quick demo ────────────────────────────────────────────────────────────────
demo_queries = [
    "What laws protect LGBTI people from workplace discrimination in Poland?",
    "What percentage of transgender people reported violence in the FRA survey?",
    "What is the HIV prevalence rate in Belgium?",
    "What is the DTP3 vaccination coverage in Romania?",
    "Tell me about queer life in Europe.",
]
print("Routing demo:")
for q in demo_queries:
    print(f"  Q: {q[:70]}")
    route_query_to_sources(q)
    print()


Routing demo:
  Q: What laws protect LGBTI people from workplace discrimination in Poland
  → Routing to: FRA (survey/experience query) + LGBT CSV backup

  Q: What percentage of transgender people reported violence in the FRA sur
  → Routing to: FRA (survey/experience query) + LGBT CSV backup

  Q: What is the HIV prevalence rate in Belgium?
  → Routing to: HIV dataset (specific HIV/AIDS query)

  Q: What is the DTP3 vaccination coverage in Romania?
  → Routing to: UNICEF dataset (immunization query)

  Q: Tell me about queer life in Europe.
  → Routing to: FRA (survey/experience query) + LGBT CSV backup



In [25]:
def retrieve_with_routing(query: str, vectorstore, k: int):
    """
    Retrieve chunks for `query`, applying metadata-based source routing.
    Falls back to unfiltered search if the filtered search returns < 2 results.
    """
    metadata_filter = route_query_to_sources(query)

    if metadata_filter:
        results = retrieve(query=query, vectorstore=vectorstore, k=k,
                           filter=metadata_filter)
        if len(results) < 2:
            print("  ⚠ Filtered results too sparse — falling back to unfiltered search.")
            results = retrieve(query=query, vectorstore=vectorstore, k=k)
    else:
        results = retrieve(query=query, vectorstore=vectorstore, k=k)

    return results


# ── Test routed retrieval on sample query ─────────────────────────────────────
print(f"Routed retrieval test for: '{SAMPLE_QUERY}'")
routed_results = retrieve_with_routing(SAMPLE_QUERY, vectorstore, k=TOP_K)
print(f"Retrieved {len(routed_results)} chunks")
print_results(query=SAMPLE_QUERY, results=routed_results)


Routed retrieval test for: 'Bisexual women daily life experience in Romania'
  → Routing to: FRA (survey/experience query) + LGBT CSV backup
Retrieved 5 chunks
Query: 'Bisexual women daily life experience in Romania'

--- Result 1 ---
Source: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\3_txt_KB\LGBT_EU\by_subset\LGBT_Survey_DailyLife\g4_b_answer_by_Bisexualwomen.txt
Question g4_b — You have been treated with less respect than other people - In the last six months, in your day-to-day life, how often have any of the following things happened to you because you are or are assumed to be lesbian, gay, bisexual and/or transgender? | Bisexual women responses in Romania: Never happened in the last sixth months: 59%, Happened only once in the last six months: 16%, 2-5 times in the last six months: 14%, 6 times or more in the last six months: 4%, Don`t know: 7%

--- Result 2 ---
Source: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\3_txt_KB\LGBT_EU\by_subset\LGBT_Survey_DailyLi

# Improve Prompting

The default `build_prompt()` uses a minimal system message.
The improved version below wraps it with richer instructions that:
- Tell the model **what kind of data** each source contains
- Ask it to **cite sources** and distinguish between PDF reports vs CSV data
- Instruct it to **acknowledge data limits** (e.g. if only CSV-structured data was retrieved)
- Prevent hallucination by being explicit about uncertainty

In [26]:
IMPROVED_SYSTEM_PROMPT = """You are an expert assistant on LGBTQI+ rights, health, and social conditions in Europe.

You answer questions using ONLY the context passages provided below. The context may come from several sources:
- **ILGA-Europe reports**: narrative policy and legal analysis, country rankings, rights legislation.
- **FRA (EU Fundamental Rights Agency) reports**: survey-based findings on discrimination, harassment, and lived experiences.
- **EU LGBT Survey (CSV)**: structured percentage data from the 2012 FRA survey (DailyLife, Discrimination, Violence, Trans, Rights).
- **HIV/AIDS dataset**: country-level statistics on HIV prevalence, deaths, and ART coverage.
- **UNICEF Immunization dataset**: vaccination coverage rates by country and vaccine.

Instructions:
1. Base your answer strictly on the provided context. Do not add outside knowledge.
2. If the context contains percentages or statistics, quote them precisely.
3. If the context is from structured CSV data (short percentage lines), synthesize the numbers into a coherent paragraph rather than listing raw data.
4. If the retrieved context is insufficient to answer the question fully, say so clearly and explain what information is missing.
5. Where relevant, note which type of source the information comes from (e.g. 'According to the ILGA-Europe report...' or 'FRA survey data shows...').
"""

def build_improved_prompt(query: str, context: str):
    """
    Wrapper around build_prompt() that substitutes the richer system message.
    Falls back to build_prompt() signature — compatible with ask().
    """
    # build_prompt returns a list of LangChain messages; we override the system content
    messages = build_prompt(query=query, context=context)
    # The first message should be the SystemMessage — replace its content
    try:
        messages[0].content = IMPROVED_SYSTEM_PROMPT
    except (IndexError, AttributeError):
        pass   # if prompt format differs, use as-is
    return messages

In [27]:
TEST_QUESTIONS = [
    # LGBT EU survey
    "What percentage of gay men in Germany experienced discrimination in the past year?",
    "How comfortable do lesbian women in France feel being open about their identity at work?",
    "What share of transgender people in Poland reported hate-motivated violence?",
    "Compare acceptance levels of same-sex couples in Sweden versus Hungary.",
    # HIV / AIDS
    "What is the HIV prevalence rate among adults in Belgium?",
    "What barriers to HIV & AIDS treatment exist in Bosnia and Herzegovina?",
    # UNICEF Immunization
    "What is the vaccination coverage rate for measles in Albania?",
    "Which Countries have the lowest DTP3 immunization rates according to UNICEF data?",
]

# Helper functions & variables

In [28]:
def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

_ensure("textstat")
_ensure("openpyxl")
_ensure("nltk")

nltk.download("punkt",      quiet=True)
nltk.download("punkt_tab",  quiet=True)

True

In [29]:
def _tokenize(text: str) -> set:
    """Lowercase word tokens, punctuation stripped."""
    return set(re.findall(r"\b[a-z]+\b", text.lower()))


def retrieval_metrics(query: str, chunks, embeddings_model) -> dict:
    """Compute retrieval-level metrics for a list of LangChain Documents."""
    texts = [c.page_content for c in chunks]

    # Duplicate ratio
    unique_texts = set(texts)
    dup_ratio = round(1 - len(unique_texts) / len(texts), 3) if texts else 0.0

    # Unique source files
    sources = [c.metadata.get("Source", "") for c in chunks]
    unique_src = len(set(sources))

    # Cosine similarity between query embedding and chunk embeddings
    try:
        q_vec  = np.array(embeddings_model.embed_query(query)).reshape(1, -1)
        c_vecs = np.array(embeddings_model.embed_documents(list(unique_texts)))
        cos_scores = cosine_similarity(q_vec, c_vecs)[0]
        cos_avg = round(float(cos_scores.mean()), 4)
    except Exception:
        cos_avg = None

    return {
        "# Chunks Retrieved": len(texts),
        "Unique Sources":     unique_src,
        "Duplicate Ratio":    dup_ratio,
        "Query Chunk Cosine Avg": cos_avg,
    }

    # Compute answer-quality metrics
def answer_metrics(query: str, answer: str, chunks) -> dict:
    no_info_phrases = [
        "don't have enough information",
        "do not have enough information",
        "cannot answer",
        "not in the context",
    ]
    answered = not any(p in answer.lower() for p in no_info_phrases)

    answer_words = _tokenize(answer)
    chunk_words  = set()
    for c in chunks:
        chunk_words |= _tokenize(c.page_content)
    query_words = _tokenize(query)

    groundedness   = round(len(answer_words & chunk_words) / len(answer_words), 3) if answer_words else 0.0
    relevancy      = round(len(answer_words & query_words) / len(query_words),  3) if query_words  else 0.0
    word_count     = len(answer.split())
    try:
        flesch = round(flesch_reading_ease(answer), 1)
    except Exception:
        flesch = None

    return {
        "Answer Length Words":  word_count,
        "Answered":             answered,
        "Groundedness":         groundedness,
        "Relevancy Score":      relevancy,
        "Flesch Reading Ease":  flesch,
    }

In [30]:
# Formatting for the Excel Output, will be used later
HEADER_FILL  = PatternFill("solid", start_color="698237", end_color="698237")   # dark green 5a7540
ALT_FILL     = PatternFill("solid", start_color="EAFACA", end_color="EAFACA")   # light green
HEADER_FONT  = Font(bold=True, color="FFFFFF", name="Arial", size=11)
BODY_FONT    = Font(name="Arial", size=10)
WRAP_ALIGN   = Alignment(wrap_text=True, vertical="top")
CENTER_ALIGN = Alignment(horizontal="center", vertical="top")
THIN_BORDER  = Border(
    left=Side(style="thin"), right=Side(style="thin"),
    top=Side(style="thin"),  bottom=Side(style="thin"),
)

# Colour coding for 'answered' column on Summary sheet
GREEN_FILL = PatternFill("solid", start_color="C6EFCE", end_color="C6EFCE")
RED_FILL   = PatternFill("solid", start_color="FFC7CE", end_color="FFC7CE")

# Parameter Sweep — Finding Best Values
Two subsections:
1. **k-sweep** — test a range of `TOP_K` values with fixed chunk settings
2. **chunk-sweep** — test paired `(CHUNK_SIZE, CHUNK_OVERLAP)` values, each requiring a fresh vectorstore

Each run writes its own `.xlsx` file to the corresponding test directory:
- `data/4_testing/range_of_k/`
- `data/4_testing/range_of_chunk/`

We format the results in identical Excel files (like `RAG_evaluation.xlsx`) so results are comparable.


## Main RAG loop function

In [31]:
def run_eval_loop(questions, vectorstore, k, embeddings_model, use_routing=True):
    summary_rows   = []
    retrieval_rows = []

    for q_idx, question in enumerate(questions, start=1):
        print(f"  [{q_idx}/{len(questions)}] {question[:75]}...")

        # Retrieve
        if use_routing:
            retrieved = retrieve_with_routing(question, vectorstore, k=k)
        else:
            retrieved = retrieve(query=question, vectorstore=vectorstore, k=k)

        ctx    = format_context(retrieved)
        prompt = build_improved_prompt(query=question, context=ctx)
        answer = ask(prompt, ctx)

        r_metrics = retrieval_metrics(question, retrieved, embeddings_model)
        a_metrics = answer_metrics(question, answer, retrieved)

        row = {"Q ID": q_idx, "Question": question, "Answer": answer}
        row.update(r_metrics)
        row.update(a_metrics)
        summary_rows.append(row)

        seen_texts = set()
        for rank, chunk in enumerate(retrieved, start=1):
            text   = chunk.page_content
            is_dup = text in seen_texts
            seen_texts.add(text)
            retrieval_rows.append({
                "Q ID":         q_idx,
                "Question":     question,
                "Rank":         rank,
                "Source":       chunk.metadata.get("Source", ""),
                "Is Duplicate": is_dup,
                "Chunk Text":   text,
            })

    return summary_rows, retrieval_rows


def write_eval_xlsx(output_path, summary_rows, retrieval_rows, param_label):
    """
    Write evaluation results to an Excel file.
    Same sheet structure as RAG_evaluation.xlsx (Summary / Retrieval_Detail / Score_Explanation).
    """
    SCORE_EXPLANATIONS = [
        {"Metric": "# Chunks Retrieved",       "Sheet": "Summary / Retrieval_Detail", "Description": "Total chunks returned per query. Controlled by TOP_K."},
        {"Metric": "Unique Sources",           "Sheet": "Summary",                    "Description": "Distinct source files in retrieved chunks. Higher = more diverse."},
        {"Metric": "Duplicate Ratio",          "Sheet": "Summary",                    "Description": "Fraction of chunks that are exact duplicates. Lower is better."},
        {"Metric": "Query Chunk Cosine Avg",   "Sheet": "Summary",                    "Description": "Mean cosine similarity between query and unique chunks. Higher = more relevant."},
        {"Metric": "Answer Length Words",      "Sheet": "Summary",                    "Description": "Word count of the answer. Very short may mean unanswerable."},
        {"Metric": "Answered",                 "Sheet": "Summary",                    "Description": "TRUE if the model gave a substantive answer."},
        {"Metric": "Groundedness",             "Sheet": "Summary",                    "Description": "Fraction of answer words that appear in retrieved chunks."},
        {"Metric": "Relevancy Score",          "Sheet": "Summary",                    "Description": "Fraction of query words that appear in the answer."},
        {"Metric": "Flesch Reading Ease",      "Sheet": "Summary",                    "Description": "Readability score. 60-70 = standard; higher = easier."},
        {"Metric": "Is Duplicate (Retrieval)", "Sheet": "Retrieval_Detail",           "Description": "TRUE if this chunk text was already seen at a higher rank."},
    ]

    df_summary   = pd.DataFrame(summary_rows)
    df_retrieval = pd.DataFrame(retrieval_rows)
    df_scores    = pd.DataFrame(SCORE_EXPLANATIONS)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df_summary.to_excel(writer,   sheet_name="Summary",           index=False)
        df_retrieval.to_excel(writer, sheet_name="Retrieval_Detail",  index=False)
        df_scores.to_excel(writer,    sheet_name="Score_Explanation", index=False)

    # ── Apply formatting ──────────────────────────────────────────────────────
    wb  = load_workbook(output_path)

    def _style_sheet(ws, col_widths, wrap_cols=None):
        wrap_cols = wrap_cols or []
        for row_idx, row in enumerate(ws.iter_rows(), start=1):
            for cell in row:
                cell.font   = HEADER_FONT if row_idx == 1 else BODY_FONT
                cell.border = THIN_BORDER
                if row_idx == 1:
                    cell.fill      = HEADER_FILL
                    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
                else:
                    cell.alignment = WRAP_ALIGN if cell.column_letter in wrap_cols else Alignment(vertical="top")
                    if row_idx % 2 == 0:
                        cell.fill = ALT_FILL
        for col_letter, width in col_widths.items():
            ws.column_dimensions[col_letter].width = width
        ws.freeze_panes = "A2"

    _style_sheet(wb["Summary"],
        col_widths={"A":6,"B":55,"C":70,"D":18,"E":16,"F":16,"G":22,"H":18,"I":12,"J":18,"K":18,"L":20},
        wrap_cols=["B","C"])

    # Colour-code 'answered'
    ws_sum = wb["Summary"]
    answered_col = [c.column for c in ws_sum[1] if c.value == "answered"]
    if answered_col:
        for row in ws_sum.iter_rows(min_row=2, min_col=answered_col[0], max_col=answered_col[0]):
            for cell in row:
                cell.fill = GREEN_FILL if cell.value is True else (RED_FILL if cell.value is False else cell.fill)

    _style_sheet(wb["Retrieval_Detail"],
        col_widths={"A":6,"B":55,"C":6,"D":60,"E":13,"F":80},
        wrap_cols=["B","D","F"])
    _style_sheet(wb["Score_Explanation"],
        col_widths={"A":28,"B":30,"C":90},
        wrap_cols=["C"])

    wb.save(output_path)
    print(f"  ✓  Saved: {output_path.name}  "
          f"({len(summary_rows)} questions, {len(retrieval_rows)} chunks)")


## k - Sweep range of values

Tests a list of `TOP_K` values against all test questions.
The vectorstore and chunk settings stay **fixed** — only the number of retrieved chunks changes.
Each value of k produces one `.xlsx` file in `data/4_testing/range_of_k/`.


In [32]:
# Loop through each value in `k_values`, run the full eval, and save one xlsx per k.
def sweep_k(k_values, questions, vectorstore, embeddings_model, output_dir, use_routing = True):
    print(f"K-sweep over {k_values}\n")

    for k in k_values:
        print(f"── k={k} ──────────────────────────────────────────")
        summary_rows, retrieval_rows = run_eval_loop(
            questions, vectorstore, k=k,
            embeddings_model=embeddings_model,
            use_routing=use_routing,
        )
        output_path = output_dir / f"RAG_eval_k={k}.xlsx"
        write_eval_xlsx(output_path, summary_rows, retrieval_rows, param_label=f"k={k}")
        print()

    print(f"K-sweep complete. Files written to: {output_dir}")

In [33]:
K_VALUES = [3, 5, 7, 10, 15]

# ── Run the k-sweep ───────────────────────────────────────────────────────────
# sweep_k(
#     k_values       = K_VALUES,
#     questions      = TEST_QUESTIONS,
#     vectorstore    = vectorstore,
#     embeddings_model = embeddings,
#     output_dir     = TEST_RANGE_K_DIR,
#     use_routing    = True,
# )


## k - Sweep range of Chunk size & overlap

Each pair requires **rebuilding the vectorstore from scratch** with new chunk settings,
then running the full eval. This is slow (one embed job per pair) but gives a true comparison.

| CHUNK_SIZE | CHUNK_OVERLAP | Notes |
|-----------|--------------|-------|
| 250       | 25           | Very small — good for short CSV-style lines; may lose narrative context |
| 500       | 50           | **Current default** — balanced |
| 750       | 75           | Larger context window per chunk; better for PDF prose |
| 1000      | 100          | Maximum useful size before chunks get too noisy |
| 1200      | 200          | High overlap — useful when key info spans chunk boundaries |


In [34]:
CHUNK_PAIRS = [
    (400,  35),
    (500,  50),    # current default
    (750,  75),
    (1000, 100),
    (1200, 200),
]
MAX_CHUNKS = 20000

   # Loop through each (CHUNK_SIZE, CHUNK_OVERLAP) pair.
   # For each pair, rebuilds the vectorstore in a temp directory, runs the eval, saves xlsx.
def sweep_chunks(chunk_pairs, questions, raw_documents, embeddings_model, output_dir, k = None, use_routing = True):
    _k = k if k is not None else TOP_K
    print(f"Chunk-sweep over {chunk_pairs}  (k={_k} fixed)\n")

    for chunk_size, chunk_overlap in chunk_pairs:
        label = f"chunk={chunk_size}_overlap={chunk_overlap}"
        print(f"── {label} ──────────────────────────────────────────")

        # 1. Re-chunk documents with new settings
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            add_start_index=True,
        )
        new_chunks = splitter.split_documents(raw_documents)
        def clean_chunk_text(text: str) -> str:
            if not text:
                return ""
            
            # Remove null bytes and bad chars
            text = text.replace("\x00", "")
            
            # Remove non-UTF8-safe characters
            text = text.encode("utf-8", "ignore").decode("utf-8")
            
            return text.strip()


        cleaned_chunks = []
        bad_count = 0

        for c in new_chunks:
            cleaned = clean_chunk_text(c.page_content)
            
            if cleaned:
                c.page_content = cleaned
                cleaned_chunks.append(c)
            else:
                bad_count += 1

        print(f"  Removed {bad_count} bad/empty chunks")
        print(f"  Chunks created: {len(new_chunks)}  "
              f"(avg {sum(len(c.page_content) for c in new_chunks)//max(len(new_chunks),1)} chars)")

        if len(cleaned_chunks) > MAX_CHUNKS:
            print(f"  ⚠ Truncating chunks to {MAX_CHUNKS}")
            cleaned_chunks = cleaned_chunks[:MAX_CHUNKS]
        # 2. Build a temp vectorstore in a subdirectory
        temp_vs_dir = PROJECT_ROOT / "data" / "vectorstore_temp" / label
        temp_vs_dir.mkdir(parents=True, exist_ok=True)

        temp_vs = build_vectorstore(
            documents=cleaned_chunks,
            embeddings=embeddings_model,
            persist_directory=str(temp_vs_dir),
        )
        print(f"  Vectorstore built: {temp_vs._collection.count()} vectors")

        # 3. Run eval
        summary_rows, retrieval_rows = run_eval_loop(
            questions, temp_vs, k=_k,
            embeddings_model=embeddings_model,
            use_routing=use_routing,
        )

        # 4. Add chunk config columns to summary for easy comparison
        for row in summary_rows:
            row["Chunk Size"]    = chunk_size
            row["Chunk Overlap"] = chunk_overlap

        # 5. Save xlsx
        output_path = output_dir / f"RAG_eval_{label}.xlsx"
        write_eval_xlsx(output_path, summary_rows, retrieval_rows, param_label=label)
        print()

    print(f"Chunk-sweep complete. Files written to: {output_dir}")


In [ ]:
# # ── Run the chunk-sweep ───────────────────────────────────────────────────────
# # ⚠ This rebuilds the vectorstore for each pair — may take several minutes per pair.
# sweep_chunks(
#     chunk_pairs    = CHUNK_PAIRS,
#     questions      = TEST_QUESTIONS,
#     raw_documents  = raw_documents,
#     embeddings_model = embeddings,
#     output_dir     = TEST_RANGE_CHUNK_DIR,
#     k              = TOP_K,
#     use_routing    = True,
# )
# # After running this and comparing results, we found that the best 
# # chunk size was 750 with overlap of 75 the best balance for our use case

Chunk-sweep over [(400, 35), (500, 50), (750, 75), (1000, 100), (1200, 200)]  (k=5 fixed)

── chunk=400_overlap=35 ──────────────────────────────────────────
  Removed 0 bad/empty chunks
  Chunks created: 44722  (avg 282 chars)
  ⚠ Truncating chunks to 20000


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Built vectorstore: 40000 chunks
  Vectorstore built: 40000 vectors
  [1/8] What percentage of gay men in Germany experienced discrimination in the pas...
  → Routing to: FRA (survey/experience query) + LGBT CSV backup


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  [2/8] How comfortable do lesbian women in France feel being open about their iden...
  → Routing to: FRA (survey/experience query) + LGBT CSV backup
  [3/8] What share of transgender people in Poland reported hate-motivated violence...
  → Routing to: FRA (survey/experience query) + LGBT CSV backup
  [4/8] Compare acceptance levels of same-sex couples in Sweden versus Hungary....
  → Routing to: ILGA (policy/rights query) + LGBT CSV backup
  [5/8] What is the HIV prevalence rate among adults in Belgium?...
  → Routing to: HIV dataset (specific HIV/AIDS query)
  ⚠ Filtered results too sparse — falling back to unfiltered search.
  [6/8] What barriers to HIV & AIDS treatment exist in Bosnia and Herzegovina?...
  → Routing to: HIV dataset (specific HIV/AIDS query)
  ⚠ Filtered results too sparse — falling back to unfiltered search.
  [7/8] What is the vaccination coverage rate for measles in Albania?...
  → Routing to: UNICEF dataset (immunization query)
  ⚠ Filtered results too sparse —

...

  [2/8] How comfortable do lesbian women in France feel being open about their iden...
  → Routing to: FRA (survey/experience query) + LGBT CSV backup
  [3/8] What share of transgender people in Poland reported hate-motivated violence...
  → Routing to: FRA (survey/experience query) + LGBT CSV backup
  [4/8] Compare acceptance levels of same-sex couples in Sweden versus Hungary....
  → Routing to: ILGA (policy/rights query) + LGBT CSV backup
  [5/8] What is the HIV prevalence rate among adults in Belgium?...
  → Routing to: HIV dataset (specific HIV/AIDS query)
  [6/8] What barriers to HIV & AIDS treatment exist in Bosnia and Herzegovina?...
  → Routing to: HIV dataset (specific HIV/AIDS query)
  [7/8] What is the vaccination coverage rate for measles in Albania?...
  → Routing to: UNICEF dataset (immunization query)
  ⚠ Filtered results too sparse — falling back to unfiltered search.
  [8/8] Which Countries have the lowest DTP3 immunization rates according to UNICEF...
  → Routing to:

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Built vectorstore: 14820 chunks
  Vectorstore built: 14820 vectors
  [1/8] What percentage of gay men in Germany experienced discrimination in the pas...
  → Routing to: FRA (survey/experience query) + LGBT CSV backup


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  [2/8] How comfortable do lesbian women in France feel being open about their iden...
  → Routing to: FRA (survey/experience query) + LGBT CSV backup
  [3/8] What share of transgender people in Poland reported hate-motivated violence...
  → Routing to: FRA (survey/experience query) + LGBT CSV backup
  [4/8] Compare acceptance levels of same-sex couples in Sweden versus Hungary....
  → Routing to: ILGA (policy/rights query) + LGBT CSV backup
  [5/8] What is the HIV prevalence rate among adults in Belgium?...
  → Routing to: HIV dataset (specific HIV/AIDS query)
  [6/8] What barriers to HIV & AIDS treatment exist in Bosnia and Herzegovina?...
  → Routing to: HIV dataset (specific HIV/AIDS query)
  [7/8] What is the vaccination coverage rate for measles in Albania?...
  → Routing to: UNICEF dataset (immunization query)
  ⚠ Filtered results too sparse — falling back to unfiltered search.
  [8/8] Which Countries have the lowest DTP3 immunization rates according to UNICEF...
  → Routing to:

# RAG Self-Evaluation

In [36]:
print(f"Running evaluation on {len(TEST_QUESTIONS)} questions...")

Running evaluation on 8 questions...


In [37]:
# Re-use the same embedding model that was used to build the vectorstore
# embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

summary_rows   = []   # one dict per question  → Sheet 1
retrieval_rows = []   # one dict per chunk     → Sheet 2

for q_idx, question in enumerate(TEST_QUESTIONS, start=1):
    print(f"[{q_idx}/{len(TEST_QUESTIONS)}] {question[:80]}...")

    # 1. Retrieve
    chunks   = retrieve(query=question, vectorstore=vectorstore, k=TOP_K)
    ctx      = format_context(chunks)

    # 2. Generate answer
    prompt   = build_prompt(query=question, context=ctx)
    answer   = ask(prompt, ctx)

    # 3. Compute metrics
    r_metrics = retrieval_metrics(question, chunks, embeddings)
    a_metrics = answer_metrics(question, answer, chunks)

    # 4. Accumulate summary row
    row = {"Q ID": q_idx, "Question": question, "Answer": answer}
    row.update(r_metrics)
    row.update(a_metrics)
    summary_rows.append(row)

    # 5. Accumulate chunk-level rows
    seen_texts = set()
    for rank, chunk in enumerate(chunks, start=1):
        text = chunk.page_content
        is_dup = text in seen_texts
        seen_texts.add(text)
        retrieval_rows.append({
            "Q ID":      q_idx,
            "Question":  question,
            "Rank":      rank,
            "Source":    chunk.metadata.get("Source", ""),
            "Is Duplicate": is_dup,
            "Chunk Text": text,
        })

[1/8] What percentage of gay men in Germany experienced discrimination in the past yea...
[2/8] How comfortable do lesbian women in France feel being open about their identity ...
[3/8] What share of transgender people in Poland reported hate-motivated violence?...
[4/8] Compare acceptance levels of same-sex couples in Sweden versus Hungary....
[5/8] What is the HIV prevalence rate among adults in Belgium?...
[6/8] What barriers to HIV & AIDS treatment exist in Bosnia and Herzegovina?...
[7/8] What is the vaccination coverage rate for measles in Albania?...
[8/8] Which Countries have the lowest DTP3 immunization rates according to UNICEF data...


In [38]:
OUTPUT_PATH_EVAL = PROJECT_ROOT / "RAG_evaluation.xlsx"

# ── Dataframes ───────────────────────────────────────────────────────────────
df_summary   = pd.DataFrame(summary_rows)
df_retrieval = pd.DataFrame(retrieval_rows)

SCORE_EXPLANATIONS = [
    {"Metric": "# Chunks Retrieved",       "sheet": "Summary / Retrieval_Detail", "Description": "Total number of chunks returned by the vector store for this query. Controlled by TOP_K."},
    {"Metric": "Unique Sources",           "sheet": "Summary",                    "Description": "Number of distinct source files among the retrieved chunks. Higher = more diverse retrieval."},
    {"Metric": "Duplicate Ratio",          "sheet": "Summary",                    "Description": "Fraction of retrieved chunks that are exact-text duplicates. 0 = no duplicates, 1 = all duplicates. Lower is better."},
    {"Metric": "Query Chunk Cosine Avg",   "sheet": "Summary",                    "Description": "Mean cosine similarity (0–1) between the query embedding and each unique chunk embedding. Higher means chunks are semantically closer to the query."},
    {"Metric": "Answer Length Words",      "sheet": "Summary",                    "Description": "Word count of the generated answer. Very short answers may indicate the model couldn't answer."},
    {"Metric": "Answered",                 "sheet": "Summary",                    "Description": "TRUE if the LLM returned a substantive answer; FALSE if it said it lacks enough information."},
    {"Metric": "Groundedness",             "sheet": "Summary",                    "Description": "Fraction of words in the answer that also appear in the retrieved chunks. Higher = answer is more grounded in retrieved evidence."},
    {"Metric": "Relevancy Score",          "sheet": "Summary",                    "Description": "Fraction of query words that appear in the answer. Higher = answer directly addresses the question."},
    {"Metric": "Flesch Reading Ease",      "sheet": "Summary",                    "Description": "Flesch Reading Ease score. 60–70 = standard; higher = easier to read; lower = more complex text."},
    {"Metric": "Is Duplicate (Retrieval)", "sheet": "Retrieval_Detail",           "Description": "TRUE if this chunk's text was already seen at a higher rank for the same question."},
]
df_scores = pd.DataFrame(SCORE_EXPLANATIONS)

# ── Write raw data via pandas ExcelWriter ────────────────────────────────────
with pd.ExcelWriter(OUTPUT_PATH_EVAL, engine="openpyxl") as writer:
    df_summary.to_excel(writer,   sheet_name="Summary",           index=False)
    df_retrieval.to_excel(writer, sheet_name="Retrieval_Detail",  index=False)
    df_scores.to_excel(writer,    sheet_name="Score_Explanation", index=False)

# ── Apply formatting with openpyxl ───────────────────────────────────────────
wb = load_workbook(OUTPUT_PATH_EVAL)

In [39]:
def _style_sheet(ws, col_widths: dict, wrap_cols: list = None):
    wrap_cols = wrap_cols or []
    for row_idx, row in enumerate(ws.iter_rows(), start=1):
        for cell in row:
            cell.font   = HEADER_FONT if row_idx == 1 else BODY_FONT
            cell.border = THIN_BORDER
            if row_idx == 1:
                cell.fill      = HEADER_FILL
                cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            else:
                if cell.column_letter in wrap_cols:
                    cell.alignment = WRAP_ALIGN
                else:
                    cell.alignment = Alignment(vertical="top")
                if row_idx % 2 == 0:
                    cell.fill = ALT_FILL
    for col_letter, width in col_widths.items():
        ws.column_dimensions[col_letter].width = width
    ws.freeze_panes = "A2"


# -- Summary sheet --
ws_sum = wb["Summary"]
ws_sum.row_dimensions[1].height = 30
_style_sheet(ws_sum,
    col_widths={"A": 6, "B": 55, "C": 70, "D": 18, "E": 16,
                "F": 16, "G": 22, "H": 18, "I": 12, "J": 18, "K": 18, "L": 20},
    wrap_cols=["B", "C"]
)
# Colour-code 'answered' column (col I = index 9, 1-based)
answered_col = [c.column for c in ws_sum[1] if c.value == "answered"]
if answered_col:
    col_letter = get_column_letter(answered_col[0])
    for row in ws_sum.iter_rows(min_row=2, min_col=answered_col[0], max_col=answered_col[0]):
        for cell in row:
            if cell.value is True:
                cell.fill = GREEN_FILL
            elif cell.value is False:
                cell.fill = RED_FILL

# -- Retrieval Detail sheet --
ws_ret = wb["Retrieval_Detail"]
_style_sheet(ws_ret,
    col_widths={"A": 6, "B": 55, "C": 6, "D": 60, "E": 13, "F": 80},
    wrap_cols=["B", "D", "F"]
)

# -- Score Explanation sheet --
ws_exp = wb["Score_Explanation"]
_style_sheet(ws_exp,
    col_widths={"A": 28, "B": 30, "C": 90},
    wrap_cols=["C"]
)
for row in ws_exp.iter_rows(min_row=2):
    ws_exp.row_dimensions[row[0].row].height = 40

wb.save(OUTPUT_PATH_EVAL)
print(f"   Saved: {OUTPUT_PATH_EVAL}")
print(f"   Sheets: Summary ({len(summary_rows)} rows)  "
      f"| Retrieval_Detail ({len(retrieval_rows)} rows)  "
      f"| Score_Explanation ({len(SCORE_EXPLANATIONS)} rows)")

   Saved: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\RAG_evaluation.xlsx
   Sheets: Summary (8 rows)  | Retrieval_Detail (40 rows)  | Score_Explanation (10 rows)


In [40]:
display_cols = [
    "Q ID", "Question", "Unique Sources", "Duplicate Ratio", "Query Chunk Cosine Avg","Answered", "Groundedness", "Relevancy Score", "Flesch Reading Ease",
]

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", "{:.3f}".format)

df_display = df_summary[display_cols].copy()
df_display["Question"] = df_display["Question"].str[:55] + "..."

df_display.head(5)

,Q ID,Question,Unique Sources,Duplicate Ratio,Query Chunk Cosine Avg,Answered,Groundedness,Relevancy Score,Flesch Reading Ease
0,1,What percentage of gay men in Germany experienced discr...,2,0.800,0.718,True,0.500,0.833,46.600
1,2,How comfortable do lesbian women in France feel being o...,2,0.800,0.697,True,0.391,0.867,22.500
2,3,What share of transgender people in Poland reported hat...,2,0.800,0.706,False,0.222,0.000,61.200
3,4,Compare acceptance levels of same-sex couples in Sweden...,2,0.800,0.663,False,0.333,0.000,61.200
4,5,What is the HIV prevalence rate among adults in Belgium...,2,0.800,0.574,False,0.000,0.000,61.200
